# DataSage 0/4 — W&B Training Data Export

Exports training metrics from 3 GRPO runs. **No GPU required.**

**Output:** `wandb_training_data.json`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/demo/inference/00_wandb_export.ipynb)

In [ ]:
!pip install -q wandb numpy

In [ ]:
import os, json
import numpy as np

try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

assert os.environ.get("WANDB_API_KEY"), "Set WANDB_API_KEY"
print(f"WANDB_API_KEY: ...{os.environ['WANDB_API_KEY'][-6:]}")

In [ ]:
WANDB_PROJECT = "ricalanis/datasage"
WANDB_RUNS = {
    "cleaning": "xuwyjpe6",
    "enrichment": "orww3s2q",
    "answering": "2mltqk5w",
}

In [ ]:
import wandb

api = wandb.Api()
wandb_data = {}

for task, run_id in WANDB_RUNS.items():
    print(f"\n{'='*60}")
    print(f"Fetching: {task} ({run_id})")
    print(f"{'='*60}")

    try:
        run = api.run(f"{WANDB_PROJECT}/{run_id}")
        print(f"  Name: {run.name} | State: {run.state}")

        history = [dict(row) for row in run.scan_history()]
        print(f"  History rows: {len(history)}")

        # Build metric series (numeric only)
        metrics = {}
        all_keys = set()
        for row in history:
            all_keys.update(row.keys())

        for key in all_keys:
            values = [row.get(key) for row in history if row.get(key) is not None]
            if values and all(isinstance(v, (int, float)) for v in values):
                metrics[key] = {
                    "values": [float(v) for v in values],
                    "min": float(min(values)),
                    "max": float(max(values)),
                    "mean": float(np.mean(values)),
                    "final": float(values[-1]),
                    "n_points": len(values),
                }

        wandb_data[task] = {
            "run_id": run_id,
            "run_name": run.name,
            "state": run.state,
            "config": dict(run.config) if run.config else {},
            "summary": {k: v for k, v in run.summary.items() if isinstance(v, (int, float, str, bool))},
            "metrics": metrics,
            "n_history_rows": len(history),
        }

        for key in sorted(metrics.keys()):
            if any(kw in key.lower() for kw in ["reward", "loss"]):
                m = metrics[key]
                print(f"  {key}: mean={m['mean']:.4f}, final={m['final']:.4f}")

    except Exception as e:
        print(f"  ERROR: {e}")
        wandb_data[task] = {"run_id": run_id, "error": str(e)}

print(f"\nDone. Tasks: {list(wandb_data.keys())}")

In [ ]:
def _json_safe(obj):
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

with open("wandb_training_data.json", "w") as f:
    json.dump(wandb_data, f, indent=2, default=_json_safe)

print(f"Saved: wandb_training_data.json ({os.path.getsize('wandb_training_data.json') / 1024:.1f} KB)")

if IN_COLAB:
    from google.colab import files
    files.download("wandb_training_data.json")